<a href="https://colab.research.google.com/github/argenviahouse-sys/Procesamiento-del-Habla/blob/main/Intro_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings

## Instalación de paquetes necesarios para este notebook

In [ ]:
!pip install --upgrade gensim numpy scipy

Si surgen conflictos en Colab luego de correr el bloque de código anterior, considera reiniciar el entorno de Colab (Runtime → Restart runtime) después de instalar los paquetes.

Cargamos las librerías que vamos a utilizar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from numpy import dot
from numpy.linalg import norm

## Similitud del coseno


Los embeddings son como una forma matemática de representar palabras, que permite a las computadoras "entender" el lenguaje de una manera más parecida a cómo lo hacen los humanos.


Siendo A y B dos vectores,

$\cos\theta = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$

Explicación de los símbolos:

* cosθ: El coseno del ángulo θ entre los dos vectores.
* A y B: Los dos vectores que se están comparando.
* A · B: El producto punto (o producto escalar) de los vectores A y B.
* ||A|| y ||B||: Las normas (o magnitudes) de los vectores A y B, respectivamente.

### ¿Qué hace la similitud del coseno?

La similitud del coseno es una medida de cuán similares son dos vectores en un espacio vectorial. El valor de cosθ varía entre -1 y 1:

* cosθ = 1: Los vectores son completamente paralelos (misma dirección).
* cosθ = -1: Los vectores son completamente antiparalelos (direcciones opuestas).
* cosθ = 0: Los vectores son ortogonales (perpendiculares).


**Usos comunes:**

* Recuperación de información: Para encontrar documentos relevantes en una base de datos.
* Recomendación de sistemas: Para sugerir productos o contenido a los usuarios.
* Análisis de clusters: Para agrupar datos similares.


Abajo se desarrolla la función *cosine_similarity* que implementa la fórmula:


$\cos\theta = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$


In [ ]:

# Función para calcular la similitud del coseno
def cosine_similarity(vec1, vec2):
    return dot(vec1, vec2) / (norm(vec1) * norm(vec2))


## Word Embedding

### modelo embedding

Vamos a aplicar el concepto de términos similares. Para ello vamos a vectorizar las palabras usando el embedding Glove. Primero es necesario indicarle a gensim que descarge dicho embedding.


In [ ]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-100")

# model = api.load('word2vec-google-news-300') # OJO QUE AQUI DESCARGA UN GIGA Y ALGO

[==================================================] 100.0% 128.1/128.1MB downloaded


### model.most_similar

Una vez creado el modelo de embedding, podemos usar funciones como most_similar para traer los tokens más similares o cercanos según su distancia. Observa los ejemplos.

In [ ]:

print(model.most_similar("king", topn=5))  # Ejemplo de uso


[('prince', 0.7682328820228577), ('queen', 0.7507690787315369), ('son', 0.7020888328552246), ('brother', 0.6985775232315063), ('monarch', 0.6977890729904175)]


In [ ]:

print(model.most_similar("woman", topn=5))


[('girl', 0.8472671508789062), ('man', 0.832349419593811), ('mother', 0.827568769454956), ('boy', 0.7720510363578796), ('she', 0.7632068395614624)]


In [ ]:

print(model.most_similar("banana", topn=5))


[('coconut', 0.7097253203392029), ('mango', 0.7054824829101562), ('bananas', 0.6887733340263367), ('potato', 0.6629636287689209), ('pineapple', 0.6534532308578491)]


In [ ]:

print(model.most_similar("apple", topn=5))


[('microsoft', 0.7449405193328857), ('ibm', 0.6821643114089966), ('intel', 0.6778088212013245), ('software', 0.6775422692298889), ('dell', 0.6741442680358887)]


### Analogías

Puedes resolver analogías como "A es a B como C es a _" haciendo A + C - B.

In [ ]:
# us is to burger as italy is to ___?
model.most_similar(positive=["Argentina", "burger"], negative=["USA"], topn=1)

KeyError: "Key 'Argentina' not present in vocabulary"

In [ ]:
# us is to burger as italy is to ___?
model.most_similar(positive=["Mexico", "burger"], negative=["USA"], topn=1)

In [ ]:
# us is to burger as italy is to ___?
model.most_similar(positive=["Italy", "burger"], negative=["USA"], topn=1)

### Palabra que no corresponda a una lista

Podemos usar la función doesnt_match para encontrar un término que no esté relacionado a los otros.

In [ ]:
model.doesnt_match(["summer", "fall", "spring", "air"])

### obtener el vector embedding de uno o varios tokens

Dada unas palabras de ejemplo que almacenamos en words, vamos a traer los vectores embeddings que las representan

In [ ]:

# Palabras de ejemplo
words = ['king', 'queen', 'man', 'woman', 'apple', 'banana']

# Obtener los vectores para esas palabras
word_vectors = np.array([model[word] for word in words])


In [ ]:
word_vectors.shape

### Graficar los embeddins en 2D

In [ ]:

# Reducir las dimensiones para visualización
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)


In [ ]:

# Graficar las palabras en el espacio 2D reducido
plt.figure(figsize=(8, 6))
for i, word in enumerate(words):
    plt.scatter(word_vectors_2d[i, 0], word_vectors_2d[i, 1], marker='o', color='blue')
    plt.text(word_vectors_2d[i, 0] + 0.01, word_vectors_2d[i, 1], word, fontsize=12)

plt.title('Word Embeddings Visualización 2D (PCA)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True)
plt.show()



### similitud del coseno de nuestras palabras ejemplo

In [ ]:
# Ejemplo de similitud del coseno
word_pairs = [('king', 'queen'), ('man', 'woman'), ('apple', 'banana'), ('king', 'man')]

similarities = []
for word1, word2 in word_pairs:
    vec1 = model[word1]
    vec2 = model[word2]
    sim = cosine_similarity(vec1, vec2)
    similarities.append((word1, word2, sim))


In [ ]:

import pandas as pd

# Crear un dataframe con las similitudes del coseno
df_similarities = pd.DataFrame(similarities, columns=["Word 1", "Word 2", "Cosine Similarity"])

df_similarities


,Word 1,Word 2,Cosine Similarity
0,king,queen,0.750769
1,man,woman,0.832349
2,apple,banana,0.505447
3,king,man,0.511868


# Tarea en casa

## a) Usa otro modelo de embedding

Utiliza otro modelo de embedding y corre el mismo código anterior. Puedes copiar y pegar las celdas de arriba, abajo. O ir probando a la par arriba.

* ¿Cuáles son las diferencias que notas?



## b) Comparando palabras

Usar el modelo para hacer un ranking de las siguientes 15 palabras según su similitud con las palabras "man" y "woman". Para cada par, imprime su similitud.

In [ ]:
words = [
"wife",
"husband",
"child",
"queen",
"king",
"man",
"woman",
"birth",
"doctor",
"nurse",
"teacher",
"professor",
"engineer",
"scientist",
"president"]

In [ ]:
# prompt: Dado el modelo embedding model , calcular para cada palabra en words la similitud con man y woman. Armar un dataframe con los resultados

import pandas as pd
data = []
for word in words:
    if word in model:
        sim_man = cosine_similarity(model[word], model['man'])
        sim_woman = cosine_similarity(model[word], model['woman'])
        data.append([word, sim_man, sim_woman])
    else:
        print(f"'{word}' not in vocabulary")

df_similarity_man_woman = pd.DataFrame(data, columns=["Word", "Similarity with Man", "Similarity with Woman"])
df_similarity_man_woman

,Word,Similarity with Man,Similarity with Woman
0,wife,0.655287,0.750502
1,husband,0.653073,0.721134
2,child,0.668265,0.760176
3,queen,0.474032,0.509515
4,king,0.511868,0.365745
5,man,1.000000,0.832349
6,woman,0.832349,1.000000
7,birth,0.448753,0.594851
8,doctor,0.609216,0.633348
9,nurse,0.456239,0.613944


¿ Cuáles son las conclusiones que tienes sobre estos resultados?